# Detalhes do projeto: 

- Versão Python: 3.14.7

# 2. DESAFIO
Você acaba de ser contratado como Analista de Dados Júnior em uma empresa de varejo que possui um histórico de vendas em formato CSV e precisa de um relatório analítico para a reunião trimestral da diretoria. O time de negócios quer entender:

- Como as vendas se comportam ao longo do tempo (por mês e por trimestre);
- Quais produtos e categorias geram mais receita?
- Quais regiões têm melhor desempenho?
- Quais clientes são mais valiosos (segmentação simples por faixa de gasto)?
- Quantas vendas individuais tiveram receita acima da média geral de receita por venda?


In [1]:
# Agrupando importações necessárias para o projeto 
import os
import csv
import json
import random
import re
from datetime import datetime, timedelta
#from funcoes_dataset import carregar_dataset, inspecionar_dados, limpar_dados, criar_colunas_derivadas, calcular_metricas, segmentar_clientes, segmentar_clientes
import funcoes_dataset as fdt
from collections import defaultdict # RF05

# RF01 – Criar ou Carregar o Dataset de Vendas

Gerando sementes do dataset conforme recomendação do guia para o projeto. (Desenvolvedor(a) em IA para Análise Preditiva [T4])

Dei preferência por modularizar o código, já que não faz parte do escopo principal e não manter o código completo no mesmo arquivo do projeto principal.

In [2]:
# Importando função para gerar as sementes do dataset
from gerar_sementes import gerar_dataset_vendas

# chamando a função
gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42)

Dataset gerado com 200 registros em vendas.csv.


# RF02 – Inspecionar e Descrever os Dados

Observações: O arquivo, ou pelo menos o nome do arquivo e formato precisam existir antes de executar o comando.

Em versões futuras, criar uma função para validar a existência do arquivo antes de importar um arquivo inexistente e informar na tela para o usuário que o arquivo não existe antes de "quebrar" fluxo do código.

Importando o dataset

In [3]:
# carregar_dataset(caminho_csv="vendas.csv")

registros = fdt.carregar_dataset("vendas.csv")
fdt.inspecionar_dados(registros)


=== INSPECAO INICIAL DO DATASET ===
Total de registros: 200

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Valores ausentes por coluna:
{'id_venda': 0, 'data_venda': 0, 'cliente': 0, 'produto': 0, 'categoria': 0, 'regiao': 0, 'quantidade': 10, 'preco_unitario': 4}

Primeiros registros:
{'id_venda': '1', 'data_venda': '2025-05-21', 'cliente': 'cliente#016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': '2', 'preco_unitario': '102.9'}
{'id_venda': '2', 'data_venda': '2025-09-16', 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '3', 'data_venda': '2025-03-23', 'cliente': 'Cliente_045', 'produto': 'Tablet', 'categoria': 'Celulares', 'regiao': 'Centro-Oeste', 'quantidade': '1', 'preco_unitario': '1939.76'}
{'id_venda': '4', 'data_venda': '2025-11-06', 'cliente': 'Cliente_017', 'p

[{'id_venda': '1',
  'data_venda': '2025-05-21',
  'cliente': 'cliente#016',
  'produto': 'Mouse',
  'categoria': 'Perifericos',
  'regiao': 'Sudeste',
  'quantidade': '2',
  'preco_unitario': '102.9'},
 {'id_venda': '2',
  'data_venda': '2025-09-16',
  'cliente': 'Cliente_039',
  'produto': 'Notebook',
  'categoria': 'Computadores',
  'regiao': 'Norte',
  'quantidade': '',
  'preco_unitario': '3204.57'},
 {'id_venda': '3',
  'data_venda': '2025-03-23',
  'cliente': 'Cliente_045',
  'produto': 'Tablet',
  'categoria': 'Celulares',
  'regiao': 'Centro-Oeste',
  'quantidade': '1',
  'preco_unitario': '1939.76'},
 {'id_venda': '4',
  'data_venda': '2025-11-06',
  'cliente': 'Cliente_017',
  'produto': 'Notebook',
  'categoria': 'Computadores',
  'regiao': 'Norte',
  'quantidade': '6',
  'preco_unitario': '3864.87'},
 {'id_venda': '5',
  'data_venda': '2025-07-05',
  'cliente': 'Cliente_037',
  'produto': 'Tablet',
  'categoria': 'Celulares',
  'regiao': 'Sul',
  'quantidade': '10',
  'pre

# RF03 – Limpar e Tratar os Dados (datetime e regex)

Implemente uma função de limpeza que trate, no mínimo:
- Espaços extras em campos de texto, usando .strip();
- Datas inválidas: converter data_venda com datetime.strptime(...) dentro de um try/except e remover os registros que não puderam ser convertidos;
- Valores ausentes nos campos críticos quantidade e preco_unitario, descartando o registro quando algum deles estiver vazio;
- Tipos de dados: garantir int para quantidade e float para preco_unitario;
Ruído nos nomes de cliente, usando o módulo re: remover caracteres não alfanuméricos, padronizar o formato para - Cliente_NNN e sinalizar os registros fora do padrão;
- Relatório de limpeza: exibir no console quantos registros entraram, quantos foram removidos (por motivo) e quantos permaneceram.

In [4]:
#limpar_dados(registros)
# Salvando os dados limpos em novas variáveis 
registros_limpos, relatorio = fdt.limpar_dados(registros)


=== RELATORIO DE LIMPEZA ===
{'iniciais': 200, 'removidos_data': 4, 'removidos_nulos': 13, 'finais': 183}


# ### Verificar e Corrigir!!!!! ###

data_venda está salvando os dados separados por vírgulas, incluiu "datatime.datetime".

De onde veio isso?

R: Não é erro, é tratamento de dados do Python, vai me permitir avançar nas próximas etapas de análise

In [5]:
#converter_data(registros)

#print(registros[3]) # Testando registro "4" a posição 3

In [6]:
# # conversao de data com tratamento de erro
# try:
#     data = datetime.strptime(texto_data, "%Y-%m-%d")
# except ValueError:
#     data = None                       # marca como invalida

# # limpeza de texto com expressao regular
# nome_limpo = re.sub(r"[^A-Za-z0-9_]", "", nome.strip())

# # validacao de formato com padrao compilado
# padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)

# RF04 – Criar Colunas Derivadas com Transformações Condicionais

A partir do dataset limpo, crie no mínimo as seguintes colunas:

Coluna |              Como obter

- receita_total:       quantidade × preco_unitario (multiplicação simples por registro)

- mes:                 Componente de mês extraído do objeto datetime com .month

- mes_nome:            Nome do mês em português, a partir de um dicionário de mapeamento

- trimestre:           Trimestre no formato Q1–Q4, calculado a partir do mês com if/elif/else

- ano:                 Ano extraído do objeto datetime com .year

- faixa_receita_item:  Classificação condicional da receita do registro com if/elif/else

In [7]:
# Dicionário de mapeamento de meses
nome_mes = {1:"Janeiro", 2:"Fevereiro", 3:"Março", 4:"Abril", 5:"Maio", 6:"Junho", 7:"Julho", 8:"Agosto",9:"Setembro",10:"Outubro",11:"Novembro",12:"Dezembro",0:"N/A"}

In [8]:
# Imprimindo dados para tratamento
ler_registro = registros_limpos[0]

print(ler_registro)
print()
print(f'=== Dados Limpos do Registro {ler_registro}')
print()
print("ID:", ler_registro["id_venda"])
print("Data:", ler_registro["data_venda"])
print("Ano:", ler_registro["data_venda"].year)
print("Mês:", ler_registro["data_venda"].month)
print("Qtd:", ler_registro["quantidade"])
print("Preço:", ler_registro["preco_unitario"])

{'id_venda': '1', 'data_venda': datetime.datetime(2025, 5, 21, 0, 0), 'cliente': 'cliente016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': 2, 'preco_unitario': 102.9, 'cliente_fora_do_padrao': True}

=== Dados Limpos do Registro {'id_venda': '1', 'data_venda': datetime.datetime(2025, 5, 21, 0, 0), 'cliente': 'cliente016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': 2, 'preco_unitario': 102.9, 'cliente_fora_do_padrao': True}

ID: 1
Data: 2025-05-21 00:00:00
Ano: 2025
Mês: 5
Qtd: 2
Preço: 102.9


In [9]:
# Trecho de apoio – transformação condicional

# mes = data_venda.month
# if receita_total < 500:
#     faixa_receita_item = "Baixo Valor"
# elif receita_total < 5000:
#     faixa_receita_item = "Medio Valor"
# else:
#     faixa_receita_item = "Alto Valor"


In [10]:
# Testando fórmulas antes do looping
rf04_receita_total = registros_limpos[0]["quantidade"] * registros_limpos[0]["preco_unitario"]
rf04_mes = registros[0]["data_venda"].month
#rf04_mes_nome 
#trimestre
rf04_ano = registros_limpos[0]["data_venda"].year

print(rf04_receita_total)
print(rf04_mes)
print(nome_mes[rf04_mes])
# trimestre print(rf04_trimestre)
print(rf04_ano)

205.8
5
Maio
2025


In [11]:
# 1. Executar a transformacao dos dados limpos
registros_enriquecidos = fdt.criar_colunas_derivadas(registros_limpos)

# 2. Imprimir os primeiros 3 registros para inspecao e conferencia
print(f"Total de registros transformados: {len(registros_enriquecidos)}\n")
for reg in registros_enriquecidos[:3]:
    print(f"ID: {reg['id_venda']:<3} | Data: {reg['data_venda'].strftime('%d/%m/%Y')} | "
          f"Mês: {reg['mes_nome']} ({reg['trimestre']}) | "
          f"Receita: R$ {reg['receita_total']:>8.2f} | Faixa: {reg['faixa_receita_item']}")

Total de registros transformados: 183

ID: 1   | Data: 21/05/2025 | Mês: Maio (Q2) | Receita: R$   205.80 | Faixa: Baixo Valor
ID: 3   | Data: 23/03/2025 | Mês: Março (Q1) | Receita: R$  1939.76 | Faixa: Medio Valor
ID: 4   | Data: 06/11/2025 | Mês: Novembro (Q4) | Receita: R$ 23189.22 | Faixa: Alto Valor


# RF05 – Calcular Métricas Agregadas por Mês, Produto, Categoria e Região

O projeto deverá calcular e exibir métricas agrupadas em, no mínimo, quatro dimensões:

- Receita total, quantidade vendida e número de vendas por mês;

- Receita total por produto (Top 5, em ordem decrescente);

- Receita total por categoria;

- Receita total e ticket médio por região.

As métricas devem ser retornadas em uma estrutura composta (por exemplo, um dicionário de listas de registros) e exibidas no console de forma legível.

In [12]:
# print(calcular_metricas(registros_limpos))

# Calculando as métricas usando os registros que já passaram pelo RF04
metricas_agregadas = fdt.calcular_metricas(registros_enriquecidos)

# Imprimindo as tabelas
print("=== POR MÊS ===")
# Usando formatação de 15 espaços e alinhamento à esquerda para criar o "efeito" de "colunas" no print
print(f"{'Mês':<5} | {'Receita Total':<15} | {'Qtd.':<5} | {'Nº Vendas'}")
print("-" * 45)
for linha in metricas_agregadas["por_mes"]:
    print(f"{linha['mes']:<5} | R$ {linha['receita_total']:<12.2f} | {linha['quantidade']:<5} | {linha['n_vendas']}")
print("\n=== TOP 5 PRODUTOS ===")
print(f"{'Produto':<15} | {'Receita Total'}")
print("-" * 35)
for linha in metricas_agregadas["top_produtos"]:
    print(f"{linha['produto']:<15} | R$ {linha['receita_total']:.2f}")
print("\n=== POR REGIÃO (C/ TICKET MÉDIO) ===")
print(f"{'Região':<15} | {'Receita Total':<15} | {'Ticket Médio'}")
print("-" * 45)
for linha in metricas_agregadas["por_regiao"]:
    print(f"{linha['regiao']:<15} | R$ {linha['receita_total']:<12.2f} | R$ {linha['ticket_medio']:.2f}")

=== POR MÊS ===
Mês   | Receita Total   | Qtd.  | Nº Vendas
---------------------------------------------
1     | R$ 120866.25    | 92    | 15
2     | R$ 73895.74     | 70    | 12
3     | R$ 123869.23    | 98    | 18
4     | R$ 80636.64     | 48    | 7
5     | R$ 132080.62    | 104   | 19
6     | R$ 106534.48    | 95    | 15
7     | R$ 106667.30    | 84    | 14
8     | R$ 85790.38     | 72    | 12
9     | R$ 84081.59     | 89    | 17
10    | R$ 110865.86    | 72    | 17
11    | R$ 160297.77    | 136   | 23
12    | R$ 104760.44    | 71    | 14

=== TOP 5 PRODUTOS ===
Produto         | Receita Total
-----------------------------------
Notebook        | R$ 374174.85
Tablet          | R$ 335335.83
Smartphone      | R$ 303255.94
Monitor         | R$ 169554.67
Headset         | R$ 48368.37

=== POR REGIÃO (C/ TICKET MÉDIO) ===
Região          | Receita Total   | Ticket Médio
---------------------------------------------
Nordeste        | R$ 366321.23    | R$ 8721.93
Norte           | R$ 2999

# Não puxou imprimiu a lista completa, pesquisar como resolver e imprimir a lista completa!

# RF06 – Segmentar Clientes por Nível de Gasto

Agrupe os dados por cliente, calcule o total gasto por cada um e classifique-os em segmentos. A classificação deve usar ao menos uma função lambda aplicada com map() ou dentro de um laço.

- Gasto total acumulado | Segmento

- Abaixo de R$ 5.000,00 | Bronze

- De R$ 5.000,00 a R$ 15.000,00 | Prata

- Acima de R$ 15.000,00 | Ouro

Exiba os 10 maiores clientes e a distribuição de clientes por segmento (contagem por segmento).

In [13]:
# print(segmentar_clientes(registros_enriquecidos))

# Lembre-se de colocar a função no seu import lá no topo do projeto:
# from funcoes_dataset import segmentar_clientes

# 1. Aplicando a função
clientes_segmentados = fdt.segmentar_clientes(registros_enriquecidos)

# 2. Exibindo os 10 maiores clientes (slicing [:10])
print("=== TOP 10 CLIENTES ===")
print(f"{'Cliente':<15} | {'Total Gasto':<15} | {'Segmento'}")
print("-" * 45)
for cliente in clientes_segmentados[:10]:
    print(f"{cliente['cliente']:<15} | R$ {cliente['total_gasto']:<12.2f} | {cliente['segmento']}")

# 3. Contagem de clientes por segmento (Distribuição)
distribuicao = {"Ouro": 0, "Prata": 0, "Bronze": 0}
for cliente in clientes_segmentados:
    distribuicao[cliente["segmento"]] += 1

print("\n=== DISTRIBUIÇÃO DE CLIENTES POR SEGMENTO ===")
for segmento, quantidade in distribuicao.items():
    print(f"{segmento:<10}: {quantidade} clientes")

=== TOP 10 CLIENTES ===
Cliente         | Total Gasto     | Segmento
---------------------------------------------
Cliente_006     | R$ 65905.09     | Ouro
Cliente_036     | R$ 63433.04     | Ouro
Cliente_039     | R$ 59406.90     | Ouro
Cliente_020     | R$ 58459.54     | Ouro
Cliente_034     | R$ 52035.60     | Ouro
Cliente_009     | R$ 50827.96     | Ouro
Cliente_024     | R$ 49146.94     | Ouro
Cliente_019     | R$ 40731.05     | Ouro
Cliente_047     | R$ 38291.74     | Ouro
Cliente_012     | R$ 38111.80     | Ouro

=== DISTRIBUIÇÃO DE CLIENTES POR SEGMENTO ===
Ouro      : 35 clientes
Prata     : 14 clientes
Bronze    : 9 clientes


# RF07 – Organizar o Código em Funções Reutilizáveis

Este requisito reforça a organização do código do fluxo em funções reutilizáveis.
Funções reutilizáveis e função de ordem superior
Todo o fluxo deve estar organizado em funções com parâmetros, retorno e docstring. Além disso, implemente uma função que receba outra função como argumento e a aplique a um campo dos registros, demonstrando o conceito de função de ordem superior.


In [14]:
# Convertendo a receita total para milhares (ex: R$ 3.500,00 -> 3.50)
registros_enriquecidos = fdt.processar_coluna(
    registros_enriquecidos,
    coluna="receita_total",
    funcao_transformacao=lambda x: round(x / 1000, 2),
    nome_saida="receita_em_milhares"
)

# Classificando o volume da venda com base na quantidade
fdt.registros_enriquecidos = fdt.processar_coluna(
    registros_enriquecidos,
    coluna="quantidade",
    funcao_transformacao=lambda q: "Alto Volume" if q > 5 else "Baixo Volume",
    nome_saida="perfil_volume"
)

# --- DEPURAÇÃO / VERIFICAÇÃO ---
print("=== REGISTROS PROCESSADOS COM FUNÇÃO DE ORDEM SUPERIOR ===")
for reg in registros_enriquecidos[:3]:
    print(f"ID: {reg['id_venda']} | Receita R$: {reg['receita_total']:>8.2f} -> em Milhares: {reg['receita_em_milhares']:>5.2f}k | "
          f"Qtd: {reg['quantidade']} -> Perfil: {reg['perfil_volume']}")


=== REGISTROS PROCESSADOS COM FUNÇÃO DE ORDEM SUPERIOR ===
ID: 1 | Receita R$:   205.80 -> em Milhares:  0.21k | Qtd: 2 -> Perfil: Baixo Volume
ID: 3 | Receita R$:  1939.76 -> em Milhares:  1.94k | Qtd: 1 -> Perfil: Baixo Volume
ID: 4 | Receita R$: 23189.22 -> em Milhares: 23.19k | Qtd: 6 -> Perfil: Alto Volume


# RF08 – Exportar Resultados em CSV e JSON

O projeto deverá demonstrar leitura e escrita de arquivos em dois formatos:

- Escrever as métricas por mês em CSV com csv.writer ou csv.DictWriter;

- Escrever a segmentação de clientes em CSV;

- Escrever as estatísticas gerais em JSON com json.dump();

- Ler de volta o JSON gravado com json.load() e exibir o conteúdo, comprovando a operação.

In [ ]:
# Calculando os números para o resumo estatístico geral
receita_geral = sum(reg["receita_total"] for reg in registros_enriquecidos)
total_vendas = len(registros_enriquecidos)
ticket_medio_geral = receita_geral / total_vendas

# Vendas individuais com valor acima da média geral
vendas_acima_media = sum(1 for reg in registros_enriquecidos if reg["receita_total"] > ticket_medio_geral)

# Dicionário de estatísticas gerais
estatisticas_gerais = {
    "receita_total_geral": receita_geral,
    "total_transacoes": total_vendas,
    "ticket_medio_geral": ticket_medio_geral,
    "vendas_acima_da_media": vendas_acima_media
}

# Chamando a função de exportação
fdt.exportar_resultados(metricas_agregadas, clientes_segmentados, estatisticas_gerais)

NameError: name 'exportar_resultados' is not defined

# RF09 – Executar o Fluxo Completo (Ponto de Entrada)

O projeto deverá ter uma função main() e o bloco if __name__ == "__main__":, executando todas as etapas na ordem correta, de ponta a ponta, sem erros.

Atenção à ordem: toda a limpeza (incluindo a padronização com expressões regulares) deve ocorrer antes das transformações e das agregações. Não faça a limpeza depois de calcular as métricas.
